# Notebook 02 - Cleaning and Merging (M3)

Runs the full data engineering pipeline.

Outputs: `factors_tidy.parquet`, `factors_wide.csv`

Pipeline: Load -> Standardize -> Missing values -> Outliers -> Aggregate -> Merge -> Validate -> Export

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.utils.helpers import setup_logging
setup_logging()
sns.set_theme(style="whitegrid")
FIGURES = "../reports/figures"
os.makedirs(FIGURES, exist_ok=True)
print("Environment ready.")

## 1. Run pipeline

In [ ]:
from src.cleaning.run_pipeline import run
run()
print('Pipeline complete. Check data/processed/')

## 2. Inspect outputs

In [ ]:
from src.utils.constants import FACTORS_TIDY, FACTORS_WIDE, PLACES_CSV
tidy = pd.read_parquet(FACTORS_TIDY)
wide = pd.read_csv(FACTORS_WIDE)
places = pd.read_csv(PLACES_CSV)
print(f'Places: {len(places)}')
print(f'Tidy: {tidy.shape}')
print(f'Wide: {wide.shape}')
print(f'Factors: {sorted(tidy["factor"].unique())}')
print(f'Years: {sorted(tidy["year"].dropna().unique().astype(int).tolist())}')

## 3. Missing values per factor

In [ ]:
ms_factor = tidy.groupby('factor')['value_raw'].agg(n_total='count', n_missing=lambda x: x.isna().sum())
ms_factor['pct_missing'] = ms_factor['n_missing'] / (ms_factor['n_total'] + ms_factor['n_missing']) * 100
display(ms_factor.round(1))

## 4. Imputation, inheritance and series-break flags

In [ ]:
print('Imputed values:', tidy['is_imputed'].sum())
print('Inherited values:', tidy['is_inherited'].sum())
print('Series breaks:', tidy['series_break'].sum())
print('Gap flags:')
print(tidy['gap_flag'].value_counts())

## 5. Coverage heatmap: place x factor

In [ ]:
coverage = tidy.groupby(['place_id','factor'])['value_raw'].count().unstack(fill_value=0)
coverage_bin = (coverage > 0).astype(int)
fig, ax = plt.subplots(figsize=(12,10))
sns.heatmap(coverage_bin, ax=ax, cmap='Blues', linewidths=0.3, linecolor='#ddd')
ax.set_title('Coverage Map: Place x Factor')
ax.tick_params(axis='y', labelsize=6)
plt.tight_layout()
plt.savefig(f'{FIGURES}/02_coverage_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Validation checks

In [ ]:
valid_ids = set(places['place_id'].tolist())
unknown = set(tidy['place_id'].unique()) - valid_ids
print(f'Unknown place IDs (should be 0): {unknown}')
missing_ids = valid_ids - set(tidy['place_id'].unique())
print(f'Places with no data: {missing_ids}')

## 7. Wide table sample

In [ ]:
print('Wide columns:', list(wide.columns))
display(wide.head(10))